# Notebook 03 — Causal Feature Analysis
**Owner: Person B (CaFE) + Person C (layer evolution, patch/CLS) — Week 2**

Stage 3. Identify causally important features for flamingo vs. spoonbill, run the CaFE sanity check, and compare patch vs. CLS features across layers.

Requires: Person A to have implemented causal.compute_feature_importance().

## 1. Load class-specific activations

In [ ]:
"""Step 1a — setup, extract zip, and top up to 200 images per class.

Two-step image sourcing:
  1. Extract data/flamingo_and_spoonbill.zip if the target dirs are empty. The zip is not in the repo.
     Zip contains imagenet_val/flamingo/ and imagenet_val/spoonbill/ entries,
     so extracting to data/ yields data/imagenet_val/flamingo/ and spoonbill/.
  2. utils/load_data.py:download_images() counts existing files, hashes them,
     then streams ImageNet (val → train) to add only unique images up to 200 each.

Requires: pip install -e . (from repo root) so that src.* and utils.* resolve.
"""
import zipfile, json
from pathlib import Path
import torch
from src.config import get_config
import src.config as _cfg_mod

cfg       = get_config()
repo_root = _cfg_mod._DEFAULT_CONFIG.parent.parent

# ── Step 1: extract zip if behavior-class dirs are empty ─────────────────────
data_dir      = repo_root / cfg.data.imagenet_val_path
flamingo_dir  = data_dir / cfg.data.behavior_class_a
spoonbill_dir = data_dir / cfg.data.behavior_class_b
zip_path      = repo_root / "data" / "flamingo_and_spoonbill.zip"

n_fl = sum(1 for _ in flamingo_dir.glob("*.JPEG"))  if flamingo_dir.exists()  else 0
n_sp = sum(1 for _ in spoonbill_dir.glob("*.JPEG")) if spoonbill_dir.exists() else 0

if zip_path.exists() and (n_fl < 10 or n_sp < 10):
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(repo_root / "data")   # imagenet_val/flamingo/ → data/imagenet_val/flamingo/
    n_fl = sum(1 for _ in flamingo_dir.glob("*.JPEG"))
    n_sp = sum(1 for _ in spoonbill_dir.glob("*.JPEG"))
    print(f"Extracted: {n_fl} flamingo, {n_sp} spoonbill")
else:
    print(f"Zip already extracted: {n_fl} flamingo, {n_sp} spoonbill")

# ── Step 2: top up to cfg.data.behavior_n_images with unique streamed images ──
from utils.load_data import download_images

download_images(
    out_dir=str(data_dir),
    n=cfg.data.behavior_n_images,
)

flamingo_paths  = sorted(flamingo_dir.glob("*.JPEG"))
spoonbill_paths = sorted(spoonbill_dir.glob("*.JPEG"))
print(f"\nReady: {len(flamingo_paths)} flamingo, {len(spoonbill_paths)} spoonbill")

Zip already extracted: 50 flamingo, 57 spoonbill
flamingo: 50 images present, 39 unique hashes indexed
spoonbill: 57 images present, 44 unique hashes indexed

Streaming validation split ...


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

  saved flamingo_100.JPEG
  saved flamingo_101.JPEG
  saved spoonbill_107.JPEG
  saved flamingo_102.JPEG
  saved flamingo_103.JPEG
  saved flamingo_104.JPEG
  saved flamingo_105.JPEG
  saved spoonbill_108.JPEG
  saved flamingo_106.JPEG
  saved flamingo_107.JPEG
  saved spoonbill_109.JPEG
  saved spoonbill_110.JPEG
  saved spoonbill_111.JPEG
  saved spoonbill_112.JPEG
  saved flamingo_108.JPEG
  saved flamingo_109.JPEG
  saved flamingo_110.JPEG
  saved spoonbill_113.JPEG
  saved flamingo_111.JPEG
  saved flamingo_112.JPEG
  saved flamingo_113.JPEG
  saved spoonbill_114.JPEG
  saved flamingo_114.JPEG
  saved flamingo_115.JPEG
  saved flamingo_116.JPEG
  saved flamingo_117.JPEG
  saved spoonbill_115.JPEG
  saved flamingo_118.JPEG
  saved spoonbill_116.JPEG
  saved spoonbill_117.JPEG
  saved spoonbill_118.JPEG
  saved flamingo_119.JPEG
  saved flamingo_120.JPEG
  saved flamingo_121.JPEG
  saved spoonbill_119.JPEG
  saved flamingo_122.JPEG
  saved flamingo_123.JPEG
  saved spoonbill_120.JPE

/opt/anaconda3/envs/vit_mech/lib/python3.10/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


  saved flamingo_199.JPEG

Done. {'flamingo': '200/200', 'spoonbill': '200/200'}

Ready: 200 flamingo, 200 spoonbill


In [2]:
"""Step 1b — verify classification.

Keeps only images the base model predicts correctly.
logit indices are in cfg.data.flamingo_logit_idx / spoonbill_logit_idx
(confirmed from notebook 01: spoonbill_004.JPEG → [2.18, -0.03] → class 0 wins).
"""
from tqdm.auto import tqdm
from src.model import get_model, preprocess_image

model  = get_model()
device = next(model.parameters()).device

flamingo_idx  = cfg.data.flamingo_logit_idx
spoonbill_idx = cfg.data.spoonbill_logit_idx

def _verify(paths, expected_idx, label):
    correct = []
    for p in tqdm(paths, desc=f"Verifying {label}"):
        pixels = preprocess_image(str(p)).to(device)
        with torch.no_grad():
            logits = model.run_with_hooks(pixels, fwd_hooks=[])
        if logits[0, 0].argmax().item() == expected_idx:
            correct.append(str(p))
    print(f"  {label}: {len(correct)}/{len(paths)} correctly classified")
    return correct

flamingo_ok  = _verify(flamingo_paths,  flamingo_idx,  cfg.data.behavior_class_a)
spoonbill_ok = _verify(spoonbill_paths, spoonbill_idx, cfg.data.behavior_class_b)
print(f"\nVerified dataset: {len(flamingo_ok)} + {len(spoonbill_ok)} = "
      f"{len(flamingo_ok)+len(spoonbill_ok)} images")

/opt/anaconda3/envs/vit_mech/lib/python3.10/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


2026-06-02 18:27:47 INFO:root: Model 'facebook/dino-vitb16' is supported and passes tests.
2026-06-02 18:27:47 DEBUG:httpcore.connection: close.started
2026-06-02 18:27:47 DEBUG:httpcore.connection: close.complete
2026-06-02 18:27:47 DEBUG:httpcore.connection: close.started
2026-06-02 18:27:47 DEBUG:httpcore.connection: close.complete
2026-06-02 18:27:47 DEBUG:httpcore.connection: connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-06-02 18:27:47 DEBUG:httpcore.connection: connect_tcp.complete ret

ln_pre not set


2026-06-02 18:27:47 DEBUG:httpcore.http11: send_request_headers.started request=<Request [b'HEAD']>
2026-06-02 18:27:47 DEBUG:httpcore.http11: send_request_headers.complete
2026-06-02 18:27:47 DEBUG:httpcore.http11: send_request_body.started request=<Request [b'HEAD']>
2026-06-02 18:27:47 DEBUG:httpcore.http11: send_request_body.complete
2026-06-02 18:27:47 DEBUG:httpcore.http11: receive_response_headers.started request=<Request [b'HEAD']>
2026-06-02 18:27:48 DEBUG:httpcore.http11: receive_response_headers.complete return_value=(b'HTTP/1.1', 404, b'Not Found', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'15'), (b'Connection', b'keep-alive'), (b'Date', b'Tue, 02 Jun 2026 16:27:48 GMT'), (b'ETag', b'W/"f-mY2VvLxuxB7KhsoOdQTlMTccuAQ"'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-6a1f0484-2328657435f4ea902ccfd78e;488c5134-c4c9-494f-a019-34e2f02c315c'), (b'RateLimit', b'"resolvers";r=4979;t=116'), (b'RateLimit-Policy', b'"fixed window";"res

Loaded facebook/dino-vitb16 on mps — 86,389,248 params


Verifying flamingo:   0%|          | 0/200 [00:00<?, ?it/s]

  flamingo: 181/200 correctly classified


Verifying spoonbill:   0%|          | 0/200 [00:00<?, ?it/s]

  spoonbill: 148/200 correctly classified

Verified dataset: 181 + 148 = 329 images


In [ ]:
"""Step 1c — extract residual-stream activations for verified images.

Runs a forward pass per image and reads the hook_resid_post tensor at the
primary layer. We extract directly from the model here instead of using
the HDF5 cache because the behavior-class images may not have been present
when the cache was first built in notebook 02.

Memory: n_images × seq_len × d_model × 4 bytes
        353 × 197 × 768 × 4 ≈ 214 MB — fine to keep on CPU.
"""
hook_key = f"blocks.{cfg.sae.primary_layer}.hook_resid_post"

def _extract_acts(paths, desc):
    acts = []
    for p in tqdm(paths, desc=desc):
        pixels = preprocess_image(str(p)).to(device)
        with torch.no_grad():
            _, cache = model.run_with_cache(pixels)
        acts.append(cache[hook_key].cpu())
    return torch.cat(acts, dim=0)  # (n, seq_len, d_model)

act_flamingo  = _extract_acts(flamingo_ok,  f"Activations {cfg.data.behavior_class_a}")
act_spoonbill = _extract_acts(spoonbill_ok, f"Activations {cfg.data.behavior_class_b}")

print(f"act_flamingo:  {tuple(act_flamingo.shape)}")
print(f"act_spoonbill: {tuple(act_spoonbill.shape)}")

## 2. Per-feature importance ranking (Person A)

Coordinate: Person A runs this cell first and saves the scores.

In [ ]:
"""Step 2 — compute feature importance and select top causal features.

Two-pass approach (see src/causal.py):
  Pass 1: gradient pre-ranking — O(n_batches) backward passes, ranks all 49k features.
  Pass 2: exact ablation — only top cfg.causal.grad_top_k (200) candidates.
Runtime: ~15–30 min on A100 for 353 images.
Result cached to outputs/features/ — re-run will load from cache.
"""
from src.causal import compute_feature_importance, get_top_causal_features

feature_output_dir = repo_root / "outputs" / "features"
feature_output_dir.mkdir(parents=True, exist_ok=True)
importance_cache = feature_output_dir / f"importance_layer{cfg.sae.primary_layer}.pt"

if importance_cache.exists():
    importance = torch.load(importance_cache, map_location="cpu")
    print(f"Loaded cached importance: {importance_cache}")
else:
    importance = compute_feature_importance(
        cfg.sae.primary_layer, act_flamingo, act_spoonbill, model
    )
    torch.save(importance, importance_cache)
    print(f"Saved importance → {importance_cache}")

# Summary
n_ablated = (importance.abs() > 0).sum().item()
top5 = importance.abs().topk(5)
print(f"\nAblated candidates : {n_ablated}  (cfg.causal.grad_top_k = {cfg.causal.grad_top_k})")
print(f"Max |importance|   : {importance.abs().max():.4f}")
print(f"Top-5 features     : {[(i.item(), round(v.item(), 4)) for i, v in zip(top5.indices, top5.values)]}")

top_features = get_top_causal_features(importance, cfg.sae.primary_layer)
print(f"\nTop causal features ({cfg.causal.importance_percentile}th pct of ablated): "
      f"{len(top_features)} features")
print(f"Indices: {sorted(top_features)[:20]}{'...' if len(top_features) > 20 else ''}")

## 3. Ablation ranking plot

In [ ]:
"""Step 3 — ablation ranking plot.

Horizontal bar chart of the top-20 causally important features, annotated
with their CLIP labels from notebook 02. Requires clip_labels_*.json.
"""
from src.visualise import plot_ablation_ranking

figures_dir = repo_root / cfg.outputs.report_figures_path
figures_dir.mkdir(parents=True, exist_ok=True)

labels_path = feature_output_dir / f"clip_labels_layer{cfg.sae.primary_layer}_full.json"
if labels_path.exists():
    with open(labels_path, encoding="utf-8") as f:
        feature_labels = {int(k): v for k, v in json.load(f).items()}
else:
    feature_labels = {}
    print("Warning: CLIP labels not found — run notebook 02 first. Proceeding without labels.")

importance_dict = {fi: importance[fi].item() for fi in top_features}

fig = plot_ablation_ranking(
    importance_dict,
    feature_labels=feature_labels,
    top_n=20,
    layer=cfg.sae.primary_layer,
    save_path=str(figures_dir / f"ablation_ranking_layer{cfg.sae.primary_layer}.png"),
)
display(fig)
print(f"Saved → {figures_dir / f'ablation_ranking_layer{cfg.sae.primary_layer}.png'}")

## 4. CaFE sanity check (Person B)

Run on top 10 causally important features. Report agreement rate.

Reference: Han et al. (2025) arXiv:2509.00749

In [ ]:
from src.causal import cafe_sanity_check
from src.visualise import plot_cafe_comparison
# for feat_idx in top_features[:10]:
#     result = cafe_sanity_check(
#         cfg.sae.primary_layer, feat_idx,
#         act_flamingo, flamingo_paths, model
#     )
#     print(f"Feature {feat_idx}: agreement = {result['agreement_rate']:.2f}")
#     fig = plot_cafe_comparison(
#         result, feat_idx,
#         save_path=f"{cfg.outputs.report_figures_path}/cafe_feat{feat_idx}.png"
#     )

## 5. Patch vs. CLS feature comparison (Person C)

In [ ]:
# TODO (Person C):
# Re-run top patch retrieval with token_type="cls"
# Compare causally important features between patch and CLS.
# Key question: do CLS features encode more global/semantic concepts?
# Document in report/notes/patch_vs_cls.md

## 6. Layer-wise evolution (Person C)

In [ ]:
from src.visualise import plot_layer_evolution
# TODO (Person C):
# Run feature annotation for layers 6 and 9 (repeat notebook 02 steps)
# category_counts = {6: {...}, 9: {...}, 11: {...}}
# fig = plot_layer_evolution(
#     category_counts,
#     save_path=cfg.outputs.report_figures_path + "/layer_evolution.png"
# )

## Checkpoint

**Person A (cells 1a–3)**
- [ ] Zip extracted — flamingo/spoonbill dirs populated
- [ ] Classification verified — counts logged (expect ~150–180 correct per class)
- [ ] act_flamingo / act_spoonbill extracted (shapes printed)
- [ ] importance_layer9.pt saved to outputs/features/
- [ ] top_features non-empty (expect 20–40 features at 80th pct)
- [ ] ablation_ranking_layer9.png saved to report/figures/

**Person B (cell 4)**
- [ ] cafe_sanity_check implemented in src/causal.py
- [ ] Agreement rate reported for top-10 features
- [ ] CaFE figures saved

**Person C (cells 5–6)**
- [ ] Patch vs. CLS comparison documented in report/notes/patch_vs_cls.md
- [ ] Layer evolution figure saved